## **1. Install Selenium**

In [1]:
%%shell
# Tải file cài đặt Google Chrome bản ổn định mới nhất
wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb

# Cài đặt Google Chrome (bỏ qua cảnh báo thiếu thư viện nếu có)
dpkg -i google-chrome-stable_current_amd64.deb

# Tự động tải và cài đặt các thư viện phụ thuộc còn thiếu
apt-get install -yf

# Cài đặt thư viện Selenium
pip install selenium

(Reading database ... 117969 files and directories currently installed.)
Preparing to unpack google-chrome-stable_current_amd64.deb ...
Unpacking google-chrome-stable (145.0.7632.159-1) over (145.0.7632.159-1) ...
Setting up google-chrome-stable (145.0.7632.159-1) ...
Processing triggers for mailcap (3.70+nmu1ubuntu1) ...
Processing triggers for man-db (2.10.2-1) ...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
0 upgraded, 0 newly installed, 0 to remove and 118 not upgraded.


## **2. Import libraries**

In [6]:
import os
import re
import time
import pandas as pd

from bs4 import BeautifulSoup
from tqdm import tqdm

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## **3. Crawl story**

In [3]:
BASE_URL = "https://truyendangian.com"
START_PAGE = 1
END_PAGE = 16
TIMEOUT = 15
DETAIL_RETRIES = 3
OUTPUT_CSV = "truyendangian_dataset.csv"
OUTPUT_ZIP = "truyendangian_dataset.zip"

options = webdriver.ChromeOptions()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--blink-settings=imagesEnabled=false")
options.add_argument("window-size=1920x1080")

driver = webdriver.Chrome(options=options)

driver.implicitly_wait(5)
wait = WebDriverWait(driver, TIMEOUT)

In [7]:
def get_soup(driver, url):
    driver.get(url)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
    return BeautifulSoup(driver.page_source, "lxml")

In [8]:
story_links = []
seen = set()

for page_idx in tqdm(range(START_PAGE, END_PAGE + 1), desc="Collect story links"):

    page_url = BASE_URL if page_idx == 1 else f"{BASE_URL}/page/{page_idx}/"
    soup = get_soup(driver,page_url)

    for a in soup.select("h2 a[href]"):
        title = a.get_text(" ", strip=True)
        url = a["href"].strip().rstrip("/")

        if not title or not url:
            continue

        if "bài thơ" in title.lower():
            continue

        if url in seen:
            continue

        seen.add(url)
        story_links.append({
            "title": title,
            "url": url
        })

print("Collected:", len(story_links))

Collect story links: 100%|██████████| 16/16 [01:28<00:00,  5.53s/it]

Collected: 306


In [9]:
def scrape_story(driver, story_url, max_retries=3):
    for attempt in range(max_retries):
        try:
            soup = get_soup(driver, story_url)

            article = soup.find("article")
            for tag in article.select("ins.adsbygoogle, script, style, sup"):
                tag.decompose()

            h1 = article.find("h1")
            title = h1.get_text(" ", strip=True)

            author_source = None
            content_lines = []
            for p in article.select("p"):
                text = p.get_text(" ", strip=True)

                if "bg-des" in p.get("class", []):
                    continue

                style = p.get("style", "")
                if "text-align: right" in style:
                    author_source = text
                    break

                content_lines.append(text)

            return {
                "url": story_url,
                "title": title,
                "content": "\n".join(content_lines).strip(),
                "author_or_source": author_source
            }

        except Exception:
            time.sleep(1 + attempt)

    return None

In [10]:
sample_url = "https://truyendangian.com/cai-bong-cua-minh"
sample_item = scrape_story(driver, sample_url)
sample_item

{'url': 'https://truyendangian.com/cai-bong-cua-minh',
 'title': 'Cái bóng của mình (Truyện thiếu nhi của tác giả Thùy Dương)',
 'content': 'Cây ổi, cây sung và khóm chuối sống cạnh nhau bên bờ ao. Cây chuối hiền lành, cây ổi thì tinh nghịch, láu lỉnh và ương bướng. Cây sung nhiều tuổi hơn cả và hay lên mặt với những cây kia. Cái gì cũng bị nó chê bai. Ngay cả hai người bạn ở cạnh nó cũng không tha.\nMột hôm, cây sung nhìn cây chuối lắc đầu tỏ vẻ ái ngại:\n– Này, cậu chuối, tại sao cậu lại có những cái tai to bè nom xấu thế nhỉ. Tớ thật không thể chịu đựng nổi nữa. Đã thế mỗi lần gió to là nó lại kêu lật phật đến buồn cười. Rồi lại còn rách bươm ra\nnữa chứ, rõ chán.\nBiết sung có ý chê những tàu lá của mình, chuối ta ngẩn người ra, xấu hổ đến nỗi chín cả mặt. Nhưng không ngờ như thế nom nó lại đẹp hơn – những trái chuối chín vàng trông thật ngon mắt. Cây sung thích chí vì đã làm cho cây chuối ngượng nghịu. Nó quay sang cây ổi và tiếp tục lên giọng kẻ cả:\n– Tại sao cậu lại cao lênh kh

In [11]:
print(sample_item["content"])

Cây ổi, cây sung và khóm chuối sống cạnh nhau bên bờ ao. Cây chuối hiền lành, cây ổi thì tinh nghịch, láu lỉnh và ương bướng. Cây sung nhiều tuổi hơn cả và hay lên mặt với những cây kia. Cái gì cũng bị nó chê bai. Ngay cả hai người bạn ở cạnh nó cũng không tha.
Một hôm, cây sung nhìn cây chuối lắc đầu tỏ vẻ ái ngại:
– Này, cậu chuối, tại sao cậu lại có những cái tai to bè nom xấu thế nhỉ. Tớ thật không thể chịu đựng nổi nữa. Đã thế mỗi lần gió to là nó lại kêu lật phật đến buồn cười. Rồi lại còn rách bươm ra
nữa chứ, rõ chán.
Biết sung có ý chê những tàu lá của mình, chuối ta ngẩn người ra, xấu hổ đến nỗi chín cả mặt. Nhưng không ngờ như thế nom nó lại đẹp hơn – những trái chuối chín vàng trông thật ngon mắt. Cây sung thích chí vì đã làm cho cây chuối ngượng nghịu. Nó quay sang cây ổi và tiếp tục lên giọng kẻ cả:
– Tại sao cậu lại cao lênh khênh như thế nhỉ? Chính cậu đã làm cho nhiều cậu bé suýt ngã đấy…
Ổi tức lắm nhưng chưa biết nói thế nào. Cậu ta cúi đầu… Chợt nhìn thấy bóng mình 

In [ ]:
datasets = []

for item in tqdm(story_links, desc="Crawl detail pages"):

    result = scrape_story(
        driver=driver,
        story_url=item["url"],
    )

    if result:
        datasets.append(result)

print("Total rows:", len(datasets))

## **4. Save data to file**

In [ ]:
df = pd.DataFrame(datasets)
df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
df.head(5)

In [ ]:
print(df.shape)
print("-" * 100)

for idx in [0, 1, 2]:
    print("INDEX:", idx)
    print("URL:", df.loc[idx, "url"])
    print("TITLE:", df.loc[idx, "title"])
    print("AUTHOR/SOURCE:", df.loc[idx, "author_or_source"])
    print("CONTENT PREVIEW:")
    print(df.loc[idx, "content"])
    print("=" * 100)

In [ ]:
!zip -r $OUTPUT_ZIP $OUTPUT_CSV

In [ ]:
driver.quit()
print("Driver closed.")